# 04 Scenarios

Possible "true" Earths for testing the array designs: each published position of the deep conductor, combined with
several BDT shapes. The BDT top is the reflector imaged by matrix imaging; the base is kept for figures and for an
optional low-velocity-zone model.

Inputs: `data/velocity_model.npz`. Output: `data/scenarios.npz`, `data/scenarios.json`

In [1]:
# --- Imports ---
import json
from itertools import product

import numpy as np
import matplotlib.pyplot as plt

In [2]:
# --- Background model ---
m = np.load("../data/velocity_model.npz")
x, y, z = m["x"], m["y"], m["z"]
X, Y = np.meshgrid(x, y, indexing="ij")
AZ = 120

In [4]:
# --- Conductor position: unknown, but probably near the drill site ---
SEARCH_R = 4.0 # how far from the drill site the conductor could be (km). assumption
RADIUS = 2.0 # conductor radius (km). assumption
RING_N = [1, 6, 8] 

centres = []
for n, rr in zip(RING_N, [0, SEARCH_R / 2, SEARCH_R]):
    a = np.radians(np.arange(n) * 360 / n + (15 if n == 6 else 0))
    centres += [(round(rr * np.sin(t), 2), round(rr * np.cos(t), 2)) for t in a]
CONDUCTORS = {f"pos{i:02d}": dict(centre=c, radius=RADIUS) for i, c in enumerate(centres)}

In [5]:
# --- BDT shapes ---
SHAPES = {
    "dome":    dict(kind="dome",  top=(4.0, 6.5), base=(6.0, 9.0), R=1.25),   # conceptual model
    "steeper": dict(kind="dome",  top=(4.0, 6.5), base=(6.0, 9.0), R=0.6),    # narrower upwarp
    "thicker": dict(kind="dome",  top=(3.0, 5.5), base=(6.0, 9.0), R=1.25),   # top raised 1 km
    "block":   dict(kind="block", top=(4.0, 6.5), base=(6.0, 9.0), R=1.25, edge=0.5),   # flat top, sides over 0.5 km
    "lens":    dict(kind="lens",  mid=5.0, half=1.0, R=1.6),                  # 4-6 km at the centre, pinches out
}


def bdt_surfaces(spec, centre, radius, x, y):
    """BDT top and base depth (km) at map points, for one shape over one conductor. NaN where there is no BDT."""
    r = np.hypot(np.asarray(x) - centre[0], np.asarray(y) - centre[1])
    R = spec["R"] * radius
    if spec["kind"] == "lens":
        h = spec["half"] * np.sqrt(np.clip(1 - (r / R) ** 2, 0, 1))
        h = np.where(h > 0, h, np.nan)
        return spec["mid"] - h, spec["mid"] + h
    if spec["kind"] == "dome":
        w = 1 - np.exp(-(r / R) ** 2)
    else:
        w = np.clip((r - R) / spec["edge"] + 0.5, 0, 1)
    (t0, t1), (b0, b1) = spec["top"], spec["base"]
    return t0 + (t1 - t0) * w, b0 + (b1 - b0) * w

In [6]:
# --- Every conductor position x every shape ---
scenarios, tops, bases = {}, {}, {}
for (c, cond), (sh, spec) in product(CONDUCTORS.items(), SHAPES.items()):
    name = f"{c}_{sh}"
    tops[name], bases[name] = bdt_surfaces(spec, cond["centre"], cond["radius"], X, Y)
    scenarios[name] = dict(conductor=c, shape=sh, centre=cond["centre"], radius=cond["radius"])

np.savez_compressed("../data/scenarios.npz", x=x, y=y,
                    **{f"top_{k}": v for k, v in tops.items()}, **{f"base_{k}": v for k, v in bases.items()})
json.dump(dict(scenarios=scenarios, shapes=SHAPES, conductors=CONDUCTORS),
          open("../data/scenarios.json", "w"), indent=1)

75 scenarios


In [7]:
# --- Low-velocity zone inside the BDT ---
BUILD_LVZ = False
DROP = 0.05 # fractional Vp and Vs drop inside the zone, assumed
EDGE = 0.3 # km over which the drop ramps in at the top and base

if BUILD_LVZ:
    Z3 = z[None, None, :]
    for name in scenarios:
        t, b = np.nan_to_num(tops[name], nan=99)[..., None], np.nan_to_num(bases[name], nan=99)[..., None]
        w = np.clip(np.minimum(Z3 - t, b - Z3) / EDGE + 0.5, 0, 1)
        np.savez_compressed(f"../data/vm_lvz_{name}.npz", x=x, y=y, z=z,
                            vp=m["vp"] * (1 - DROP * w), vs=m["vs"] * (1 - DROP * w))

In [9]:
# --- Left: conductor positions in map view. Right: each shape along the node axis over the first conductor ---
s = np.linspace(-8, 8, 321)
sx, sy = s * np.sin(np.radians(AZ)), s * np.cos(np.radians(AZ))
c0 = next(iter(CONDUCTORS.values()))

with plt.style.context("default"):
    fig = plt.figure(figsize=(14, 3.8))
        ax = fig.add_subplot(1, len(SHAPES) + 1, 1)
    ax.add_patch(plt.Circle((0, 0), SEARCH_R, fill=False, ls="--", color="grey"))       # search area
    for c, cond in CONDUCTORS.items():
        ax.add_patch(plt.Circle(cond["centre"], cond["radius"], fill=False, lw=0.6, alpha=0.5))
        ax.plot(*cond["centre"], "k.", ms=4)
    ax.set(aspect="equal", xlim=(-8, 8), ylim=(-8, 8), xlabel="East (km)", ylabel="North (km)",
           title="Possible conductor positions")

    for i, (sh, spec) in enumerate(SHAPES.items()):
        ax = fig.add_subplot(1, len(SHAPES) + 1, i + 2)
        top, base = bdt_surfaces(spec, (0, 0), c0["radius"], sx, sy)
        ax.fill_between(s, top, base, color="firebrick", alpha=0.2)
        ax.plot(s, top, color="firebrick", lw=2)
        ax.set(title=sh, xlim=(-8, 8), ylim=(10, 0), xlabel="Along axis (km)")
        if i == 0:
            ax.set_ylabel("Depth (km)")
    plt.tight_layout()
    plt.show()

IndentationError: unexpected indent (4046141515.py, line 8)